In [100]:
%%capture
!pip install lightning

In [101]:
import os
import sys
from pathlib import Path

import h5py

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from torchvision import transforms
import torchvision.models as models
from torchvision.datasets import PCAM

import torchmetrics

from tqdm.notebook import tqdm

import lightning as L
from lightning import LightningModule, LightningDataModule

In [102]:
EPOCHS = 30
BATCH_SIZE = 512
SEED = 42

In [103]:
# TODO: add good transforms for training and validation datasets (roto-translation, cropping, illuminance, etc.)
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Model

In [104]:
class ResNet18Classifier(LightningModule):
    def __init__(self, num_classes: int = 2):
        super().__init__()
        self.model = models.resnet18(weights='DEFAULT')
        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)
        self.num_classes = num_classes
        
        self.criterion = nn.CrossEntropyLoss()

        self.train_acc = torchmetrics.Accuracy("binary", num_classes=num_classes)
        self.val_acc = torchmetrics.Accuracy("binary", num_classes=num_classes)
        self.test_acc = torchmetrics.Accuracy("binary", num_classes=num_classes)

    def forward(self, x):
        return self.model(x)

    def _step(self, batch, stage: str):
        images, labels = batch

        outputs = self(images)
        loss = self.criterion(outputs, labels)
        preds = torch.argmax(outputs, dim=1)

        if stage == "train":
            self.train_acc(preds, labels)
        elif stage == "val":
            self.val_acc(preds, labels)
        elif stage == "test":
            self.test_acc(preds, labels)
        else:
            raise ValueError(f"Unknown stage: {stage}")
        
        self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log(f"{stage}_acc", getattr(self, f"{stage}_acc"), on_step=False, on_epoch=True, prog_bar=True)
        
        return loss

    def training_step(self, batch, batch_idx):
        return self._step(batch, "train")
    def validation_step(self, batch, batch_idx):
        return self._step(batch, "val")
    def test_step(self, batch, batch_idx):
        return self._step(batch, "test")

    def configure_optimizers(self):
        optimizer = optim.AdamW(self.parameters(), lr=1e-3, weight_decay=0.05)
        
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.1, patience=5
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
                "interval": "epoch",
                "frequency": 1,
            },
        }

# Dataset

In [105]:
class PCAMDataset(torch.utils.data.Dataset):
    def __init__(self, input_file_path: str, label_file_path: str, transform=None):
        self.input_file_path = input_file_path
        self.label_file_path = label_file_path
        self.transform = transform

        input_files = h5py.File(self.input_file_path)["x"]
        target_files = h5py.File(self.label_file_path)["y"]

        self.images = [Image.fromarray(file).convert("RGB") for file in tqdm(input_files)]
        self.targets = [int(target_files[i, 0, 0, 0]) for i in tqdm(range(len(target_files)))]
        
    def __len__(self) -> int:
        return len(self.images)
            
    def __getitem__(self, idx: int):
        image, target = self.images[idx], torch.tensor(self.targets[idx])

        if self.transform:
            image = self.transform(image)

        return image, target

In [ ]:
train_data = PCAMDataset(
    input_file_path="/kaggle/input/metastatic-tissue-classification-patchcamelyon/pcam/training_split.h5",
    label_file_path="/kaggle/input/metastatic-tissue-classification-patchcamelyon/Labels/Labels/camelyonpatch_level_2_split_train_y.h5",
    transform=train_transform,
)

In [62]:
val_data = PCAMDataset(
    input_file_path="/kaggle/input/metastatic-tissue-classification-patchcamelyon/pcam/validation_split.h5",
    label_file_path="/kaggle/input/metastatic-tissue-classification-patchcamelyon/Labels/Labels/camelyonpatch_level_2_split_valid_y.h5",
    transform=train_transform,
)

  0%|          | 0/32768 [00:00<?, ?it/s]

  0%|          | 0/32768 [00:00<?, ?it/s]

In [63]:
test_data = PCAMDataset(
    input_file_path="/kaggle/input/metastatic-tissue-classification-patchcamelyon/pcam/test_split.h5",
    label_file_path="/kaggle/input/metastatic-tissue-classification-patchcamelyon/Labels/Labels/camelyonpatch_level_2_split_test_y.h5",
    transform=train_transform,
)

  0%|          | 0/32768 [00:00<?, ?it/s]

  0%|          | 0/32768 [00:00<?, ?it/s]

# Data Module

In [ ]:
class PCAMDataModule(LightningDataModule):
    def __init__(self, train, val, test=None, name=""):
        super().__init__()
        self.train = train
        self.val = val
        self.test = test
        
        self.name = name

    def setup(self, stage: str = None):
        self.train_dataset = self.train
        self.val_dataset = self.val
        self.test_dataset = self.test

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

In [116]:
model = ResNet18Classifier(num_classes=2)

In [117]:
model

ResNet18Classifier(
  (model): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, 

In [118]:
datamodule = PCAMDataModule(
    train=train_data,
    val=val_data,
    test=test_data
)

In [119]:
ckbs = [
    L.pytorch.callbacks.ModelCheckpoint(
            dirpath="/kaggle/working/checkpoints/",
            monitor="val_loss",
            mode="min",
            save_top_k=1,
            filename="{epoch:02d}-{val_loss:.2f}",
        ),
    L.pytorch.callbacks.LearningRateMonitor(logging_interval='epoch'),
    L.pytorch.callbacks.RichProgressBar(),
    L.pytorch.callbacks.early_stopping.EarlyStopping(
        monitor="val_loss", min_delta=0.00, patience=5, verbose=False, mode="min"
    )
]

In [120]:
tensorboard_logger = L.pytorch.loggers.TensorBoardLogger(
    save_dir="/kaggle/working/logs/",
)

In [121]:
trainer = L.Trainer(
    accelerator="auto", 
    devices="auto",
    max_epochs=EPOCHS,
    precision="16-mixed", # Use mixed precision for faster training
    num_nodes=1,
    logger=[tensorboard_logger],
    callbacks=ckbs,
    log_every_n_steps=10,
)

INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs


In [ ]:
trainer.fit(model, datamodule=datamodule)

In [125]:
trainer.test(model, datamodule=datamodule, ckpt_path="best")

INFO: Restoring states from the checkpoint path at /kaggle/working/checkpoints/epoch=00-val_loss=0.52.ckpt
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: Loaded model weights from the checkpoint at /kaggle/working/checkpoints/epoch=00-val_loss=0.52.ckpt


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │     0.776885986328125     │
│         test_loss         │    0.6869282722473145     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.6869282722473145, 'test_acc': 0.776885986328125}]